# 📉 Extended plots

Five recommended statistical plots + standalone Lander-Waterman α sweep:

① α sweep · ② Coverage vs Poisson PMF · ③ Coverage fraction vs α · ④ Gap-length distribution · ⑤ N50/N90 · ⑥ Contig-length CDF

> Requires stats dicts `s0` and `s1` produced by `shortgun_simulation.ipynb`.


In [ ]:
# !pip install matplotlib numpy
import math
import numpy as np
import matplotlib.pyplot as plt


### Helper functions


#### `compute_nx`
*Compute the Nx assembly metric (N50 when x=0.5, N90 when x=0.9).*


In [ ]:
def compute_nx(contig_lengths: list, x: float) -> int:
    """Compute the Nx assembly metric (N50 when x=0.5, N90 when x=0.9).

    Nx is the length of the shortest contig such that contigs of that length
    or longer account for at least x of the total assembled sequence.

    Args:
        contig_lengths: List of contig lengths in bp.
        x:              Fraction threshold (0 < x ≤ 1).
    Returns:
        Nx length in bp, or 0 for an empty list.
    """
    if not contig_lengths:
        return 0
    sorted_l = sorted(contig_lengths, reverse=True)
    target   = sum(sorted_l) * x
    cumsum   = 0
    for l in sorted_l:
        cumsum += l
        if cumsum >= target:
            return l
    return sorted_l[-1]


#### `extract_gaps`
*Return gap lengths (bp) between consecutive contigs.*


In [ ]:
def extract_gaps(contigs: list) -> list:
    """Return gap lengths (bp) between consecutive contigs.

    Args:
        contigs: List of (start, end) tuples in ascending order.
    Returns:
        List of positive integer gap sizes.
    """
    return [
        contigs[i + 1][0] - contigs[i][1]
        for i in range(len(contigs) - 1)
        if contigs[i + 1][0] > contigs[i][1]
    ]


# =============================================================================
# EXTENDED SUMMARY — ALL 5 RECOMMENDED PLOTS (2 × 3 FIGURE)
# =============================================================================


#### `_empirical_contig_count`
*Run one sequencing trial and return (alpha, empirical_contig_count).*


In [ ]:
def _empirical_contig_count(genome: str, num_reads: int,
                             min_len: int, max_len: int) -> tuple:
    """Run one sequencing trial and return (alpha, empirical_contig_count).

    Args:
        genome:    Reference genome to sample reads from.
        num_reads: Number of reads N for this trial.
        min_len:   Minimum read length.
        max_len:   Maximum read length (also used as L for alpha).
    Returns:
        (alpha, n_contigs) tuple.
    """
    reads      = generate_reads(genome, num_reads, min_len, max_len)
    reads_data = create_alignment(genome, reads)
    contigs    = get_contigs(reads_data)
    alpha      = compute_alpha(num_reads, max_len, len(genome))
    return alpha, len(contigs)


### Standalone α-sweep figure


#### `plot_lander_waterman_sweep`
*Plot the Lander-Waterman contig-count curve against empirical data.*


In [ ]:
def plot_lander_waterman_sweep(
        G: int = 2000,
        min_len: int = 50,
        max_len: int = 100,
        n_trials: int = 3,
        output_png: str = "lander_waterman_sweep.png") -> None:
    """Plot the Lander-Waterman contig-count curve against empirical data.

    Theory predicts:  E[# contigs] = N · e^{-α},   α = N·L / G

    The sweep varies N (number of reads) across a range chosen so that α
    spans roughly 0 → 6, giving the full shape of the exponential decay.
    Each N value is repeated `n_trials` times and the results are averaged
    to smooth out sampling noise.

    Two genome types are plotted so the reader can see exactly where the
    Poisson/uniform-start assumption breaks down for repeat-rich sequences.

    Args:
        G:          Genome length in bp (both genome types use the same G).
        min_len:    Minimum read length for sampling.
        max_len:    Maximum read length (used as L in the alpha formula).
        n_trials:   Number of independent trials per N value (averaged).
        output_png: Output filename.
    """
    L = max_len    # L used consistently in α = NL/G

    # N values chosen so α = NL/G sweeps 0.2 → 6.0
    alpha_targets = np.linspace(0.2, 6.0, 18)
    N_values      = np.round(alpha_targets * G / L).astype(int)
    N_values      = np.maximum(N_values, 1)   # guard against zero

    # Smooth theoretical curve (dense alpha grid)
    alpha_curve = np.linspace(0, 6.5, 300)

    # Pre-generate both genomes once — same G, different structure
    print("Generating genomes for α sweep …")
    genome_plain  = generate_genome(G)
    genome_repeat = generate_genome_with_repeats(G)

    emp_plain_alpha,  emp_plain_cnt  = [], []
    emp_repeat_alpha, emp_repeat_cnt = [], []

    for N in N_values:
        a_sum_p, c_sum_p = 0.0, 0.0
        a_sum_r, c_sum_r = 0.0, 0.0
        for _ in range(n_trials):
            a, c = _empirical_contig_count(genome_plain,  N, min_len, max_len)
            a_sum_p += a;  c_sum_p += c
            a, c = _empirical_contig_count(genome_repeat, N, min_len, max_len)
            a_sum_r += a;  c_sum_r += c
        emp_plain_alpha.append(a_sum_p / n_trials)
        emp_plain_cnt.append(c_sum_p / n_trials)
        emp_repeat_alpha.append(a_sum_r / n_trials)
        emp_repeat_cnt.append(c_sum_r / n_trials)
        alpha_actual = N * L / G
        print(f"  N={N:5d}  α={alpha_actual:.2f}  "
              f"plain={c_sum_p/n_trials:.1f}  "
              f"repeat={c_sum_r/n_trials:.1f}")

    # ── Figure ──────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                             gridspec_kw={"wspace": 0.35})

    C0, C1 = "#2196F3", "#F44336"

    for ax, (emp_a, emp_c), color, label, theory_scale in [
        (axes[0],
         (emp_plain_alpha,  emp_plain_cnt),  C0, "No repeats",  N_values),
        (axes[1],
         (emp_repeat_alpha, emp_repeat_cnt), C1, "With repeats", N_values),
    ]:
        # Theoretical curve: N·e^{-α} — N changes with α, so rewrite as
        # (G/L)·α·e^{-α}  (since N = αG/L)
        theory_y = (G / L) * alpha_curve * np.exp(-alpha_curve)

        ax.plot(alpha_curve, theory_y,
                color="#888", linewidth=2, linestyle="--",
                label=r"Theory: $(G/L)\,\alpha\,e^{-\alpha}$", zorder=2)

        ax.scatter(emp_a, emp_c,
                   color=color, s=55, zorder=3, edgecolors="white",
                   linewidths=0.6, label=f"Empirical ({label})")

        # Residual shading between theory and empirical
        theory_at_emp = (G / L) * np.array(emp_a) * np.exp(-np.array(emp_a))
        for xa, yc, yt in zip(emp_a, emp_c, theory_at_emp):
            ax.plot([xa, xa], [yt, yc],
                    color=color, alpha=0.25, linewidth=1, zorder=1)

        # Annotate the current simulation's operating point
        cur_alpha = N_values[len(N_values)//2] * L / G
        cur_theory = (G / L) * cur_alpha * math.exp(-cur_alpha)
        ax.annotate(f"α = {cur_alpha:.2f}\n(typical run)",
                    xy=(cur_alpha, cur_theory),
                    xytext=(cur_alpha + 0.6, cur_theory + (G/L)*0.07),
                    fontsize=8, color="#555",
                    arrowprops=dict(arrowstyle="->", color="#999",
                                   lw=0.8, connectionstyle="arc3,rad=0.2"))

        ax.set_xlabel(r"Read density  $\alpha = NL/G$", fontsize=10)
        ax.set_ylabel("Number of contigs", fontsize=10)
        ax.set_title(f"Lander-Waterman sweep — {label}", fontweight="bold")
        ax.legend(fontsize=8)
        ax.set_xlim(0, 6.8)
        ax.set_ylim(bottom=0)
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, linestyle="--", alpha=0.4)

    # Shared annotation explaining the divergence
    axes[1].text(0.97, 0.95,
                 "Dots above the curve:\nPoisson assumption violated\nby repeat structure",
                 transform=axes[1].transAxes,
                 fontsize=8, color=C1, ha="right", va="top",
                 bbox=dict(boxstyle="round,pad=0.4", fc="white",
                           ec=C1, alpha=0.85, lw=0.8))

    fig.suptitle(
        r"Lander-Waterman Model: $E[\#\,\mathrm{contigs}] = N e^{-\alpha}$",
        fontsize=13, fontweight="bold")
    plt.savefig(output_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nLander-Waterman sweep saved to '{output_png}'")


# =============================================================================
# HELPERS FOR EXTENDED PLOTS
# =============================================================================


### All-in-one extended summary (2×3 figure)


#### `generate_extended_summary_plots`
*Produce a 2 × 3 figure with all five recommended statistical plots.*


In [ ]:
def generate_extended_summary_plots(
        s0: dict,
        s1: dict,
        output_png: str = "extended_summary.png",
        sweep_trials: int = 2) -> None:
    """Produce a 2 × 3 figure with all five recommended statistical plots.

    Panel layout
    ------------
    [0,0]  ① Lander-Waterman α sweep    — theory curve vs empirical scatter
    [0,1]  ② Coverage vs Poisson(α) PMF — per-position depth histogram + PMF
    [0,2]  ③ Coverage fraction vs α     — completeness curve 1 − e^{−α}
    [1,0]  ④ Gap-length distribution    — histogram + fitted Exponential PDF
    [1,1]  ⑤ N50 / N90 bar chart        — assembly-quality grouped bars
    [1,2]  ⑥ Contig-length CDF          — empirical step-CDF with N50/N90 markers

    Args:
        s0:           Stats dict returned by shotgun_simulation_without_repeats().
        s1:           Stats dict returned by shotgun_simulation_with_repeats().
        output_png:   Output filename for the saved figure.
        sweep_trials: Trials per N value in the α-sweep panel (2 = fast,
                      raise to 4-5 for smoother empirical dots).
    """
    C0   = "#2196F3"   # blue  — no repeats
    C1   = "#F44336"   # red   — with repeats
    GRAY = "#888888"   # theory curves

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(
        "Extended Statistical Analysis — Shotgun Sequencing Simulation",
        fontsize=14, fontweight="bold", y=1.01,
    )

    G  = s0["G"]
    L  = max(s0["read_lengths"])
    a0 = s0["alpha"]
    a1 = s1["alpha"]

    # ══════════════════════════════════════════════════════════════════════════
    # [0, 0]  ① Lander-Waterman α sweep
    #
    # Theory:  E[# contigs] = N · e^{-α}  ≡  (G/L) · α · e^{-α}
    # We vary N so α sweeps 0.3 → 5.5, run sweep_trials per point, and
    # overlay the smooth theoretical curve.  Both genome types are plotted
    # so the reader sees exactly where the Poisson assumption breaks.
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[0, 0]

    alpha_targets = np.linspace(0.3, 5.5, 14)
    N_sweep       = np.maximum(np.round(alpha_targets * G / L).astype(int), 1)
    alpha_curve   = np.linspace(0, 6.2, 400)
    theory_y      = (G / L) * alpha_curve * np.exp(-alpha_curve)

    print("\n[Extended plots] Running α sweep …")
    pt_ap, pt_cp = [], []
    pt_ar, pt_cr = [], []

    for N in N_sweep:
        cp_acc = cr_acc = 0
        for _ in range(sweep_trials):
            _, cp = _empirical_contig_count(s0["genome"], N, 50, L)
            _, cr = _empirical_contig_count(s1["genome"], N, 50, L)
            cp_acc += cp
            cr_acc += cr
        alpha_n = N * L / G
        pt_ap.append(alpha_n);  pt_cp.append(cp_acc / sweep_trials)
        pt_ar.append(alpha_n);  pt_cr.append(cr_acc / sweep_trials)

    ax.plot(alpha_curve, theory_y, color=GRAY, lw=2, ls="--",
            label=r"Theory: $(G/L)\,\alpha\,e^{-\alpha}$", zorder=2)
    ax.scatter(pt_ap, pt_cp, color=C0, s=50, zorder=4,
               edgecolors="white", lw=0.7, label="No repeats")
    ax.scatter(pt_ar, pt_cr, color=C1, s=50, zorder=4, marker="^",
               edgecolors="white", lw=0.7, label="With repeats")

    # Vertical markers at each simulation's operating α
    for a_op, col in [(a0, C0), (a1, C1)]:
        ax.axvline(a_op, color=col, lw=0.9, ls=":", alpha=0.65)

    # Residual connectors (theory → empirical) for repeat genome
    theory_at_emp = (G / L) * np.array(pt_ar) * np.exp(-np.array(pt_ar))
    for xa, yc, yt in zip(pt_ar, pt_cr, theory_at_emp):
        ax.plot([xa, xa], [yt, yc], color=C1, alpha=0.20, lw=1.2, zorder=1)

    ax.text(0.97, 0.97,
            "Dots above curve:\nPoisson assumption\nviolated by repeats",
            transform=ax.transAxes, fontsize=7.5, color=C1,
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.35", fc="white",
                      ec=C1, alpha=0.85, lw=0.8))
    ax.set_xlabel(r"Read density  $\alpha = NL/G$", fontsize=9)
    ax.set_ylabel("Number of contigs", fontsize=9)
    ax.set_title("① Lander-Waterman α sweep", fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(0, 6.5);  ax.set_ylim(bottom=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, ls="--", alpha=0.35)
    print("  α sweep complete.")

    # ══════════════════════════════════════════════════════════════════════════
    # [0, 1]  ② Coverage distribution vs Poisson(α) PMF
    #
    # Under the Lander-Waterman model the number of reads starting in any
    # window of length 1 follows Poisson(α).  Per-position depth should
    # therefore follow Poisson(α) for a random genome.  Repeat-rich genomes
    # violate this: piled-up repeats produce a heavy right tail.
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[0, 1]

    bar_width = 0.38
    for offset, s, col, lbl in [
        (-bar_width / 2, s0, C0, "No repeats"),
        ( bar_width / 2, s1, C1, "With repeats"),
    ]:
        cov       = np.array(s["coverage"])
        alpha_val = s["alpha"]
        p99       = int(np.percentile(cov, 99)) + 1   # clip extreme outliers

        # Empirical depth histogram (normalised to probabilities)
        counts = np.bincount(cov, minlength=p99 + 1)[:p99 + 1]
        k_vals = np.arange(p99 + 1)
        ax.bar(k_vals + offset, counts / len(cov),
               width=bar_width, color=col, alpha=0.45,
               label=f"Empirical — {lbl}", edgecolor="none")

        # Poisson(α) PMF in log space (numerically stable for any k)
        log_pmf = (-alpha_val
                   + k_vals * np.log(alpha_val + 1e-300)
                   - np.array([math.lgamma(k + 1) for k in k_vals]))
        pmf = np.exp(log_pmf)
        ax.plot(k_vals, pmf, color=col, lw=1.8, ls="--",
                label=fr"Poisson($\alpha$={alpha_val:.2f})")

    ax.set_xlabel("Read depth at each position", fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.set_title("② Coverage distribution vs Poisson(α) PMF", fontweight="bold")
    ax.legend(fontsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, ls="--", alpha=0.35)

    # ══════════════════════════════════════════════════════════════════════════
    # [0, 2]  ③ Genome coverage fraction vs α
    #
    # Theory:  P(position covered) = 1 − e^{−α}
    # This is the Lander-Waterman completeness curve.  The empirical fraction
    # of positions with depth > 0 should track the curve for random genomes.
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[0, 2]

    alpha_fine  = np.linspace(0, 8, 500)
    theory_comp = (1 - np.exp(-alpha_fine)) * 100

    ax.plot(alpha_fine, theory_comp, color=GRAY, lw=2, ls="--",
            label=r"Theory: $1 - e^{-\alpha}$")

    # 95% and 99% horizontal guide lines
    for pct in [95, 99]:
        ax.axhline(pct, color="#CCCCCC", lw=1, ls=":")
        ax.text(7.85, pct + 0.8, f"{pct}%",
                fontsize=7.5, color="#AAAAAA", va="bottom")

    # Empirical operating points
    for s, col, lbl in [(s0, C0, "No repeats"), (s1, C1, "With repeats")]:
        cov      = np.array(s["coverage"])
        emp_pct  = float(np.sum(cov > 0)) / len(cov) * 100
        ax.scatter([s["alpha"]], [emp_pct],
                   color=col, s=90, zorder=5,
                   edgecolors="white", lw=0.8, label=f"Empirical — {lbl}")
        ax.annotate(f"{emp_pct:.1f}%",
                    xy=(s["alpha"], emp_pct),
                    xytext=(s["alpha"] + 0.2, emp_pct - 5),
                    fontsize=8, color=col,
                    arrowprops=dict(arrowstyle="->", color=col,
                                   lw=0.7, connectionstyle="arc3,rad=0.2"))

    # How many more reads are needed to reach 99%?
    for s, col in [(s0, C0), (s1, C1)]:
        alpha_99 = -math.log(0.01)           # ≈ 4.605
        N_99     = int(math.ceil(alpha_99 * G / L))
        ax.annotate("",
                    xy=(alpha_99, 99),
                    xytext=(s["alpha"], 99),
                    arrowprops=dict(arrowstyle="->", color=col,
                                   lw=0.7, linestyle="dotted"),
                    annotation_clip=False)

    ax.set_xlabel(r"$\alpha = NL/G$", fontsize=9)
    ax.set_ylabel("Genome covered (%)", fontsize=9)
    ax.set_title("③ Coverage fraction vs α", fontweight="bold")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 8.2);  ax.set_ylim(0, 106)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, ls="--", alpha=0.35)

    # ══════════════════════════════════════════════════════════════════════════
    # [1, 0]  ④ Gap-length distribution vs fitted Exponential
    #
    # Inter-contig gaps should follow Exp(1/μ_gap) under the uniform-start
    # model.  Deviations — particularly in the repeat genome — reveal that
    # the Poisson assumption no longer holds.
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[1, 0]

    any_gap_data = False
    for s, col, lbl in [(s0, C0, "No repeats"), (s1, C1, "With repeats")]:
        gaps = extract_gaps(s["contigs"])
        if not gaps:
            continue
        any_gap_data = True
        gaps_arr = np.array(gaps, dtype=float)
        mean_gap = gaps_arr.mean()
        lam      = 1.0 / mean_gap                # MLE rate parameter

        # Histogram (density=True so it overlays with the PDF)
        n_bins = min(25, max(5, len(gaps)))
        ax.hist(gaps_arr, bins=n_bins, density=True,
                color=col, alpha=0.40, edgecolor="white", lw=0.5,
                label=f"Empirical — {lbl}")

        # Fitted Exponential PDF:  f(x) = λ · e^{−λx}
        x_fit = np.linspace(0, gaps_arr.max() * 1.05, 400)
        ax.plot(x_fit, lam * np.exp(-lam * x_fit), color=col, lw=2, ls="--",
                label=fr"Exp fit ($\lambda$={lam:.4f}, μ={mean_gap:.0f} bp)")

        # Vertical mean marker
        ax.axvline(mean_gap, color=col, lw=1, ls=":", alpha=0.75)

    if not any_gap_data:
        ax.text(0.5, 0.5, "No gaps detected\n(genome fully covered)",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, color=GRAY)

    ax.set_xlabel("Gap length (bp)", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_title("④ Gap-length distribution vs Exponential fit", fontweight="bold")
    ax.legend(fontsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, ls="--", alpha=0.35)

    # ══════════════════════════════════════════════════════════════════════════
    # [1, 1]  ⑤ N50 / N90 / Max / Mean bar chart
    #
    # N50 is the standard bioinformatics assembly quality metric: the length L
    # such that 50% of the total assembled sequence lies in contigs ≥ L.
    # Lower N50 in the repeat genome directly quantifies fragmentation.
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[1, 1]

    metrics  = ["N50", "N90", "Max length", "Mean length"]
    vals_0   = [
        compute_nx(s0["contig_lengths"], 0.50),
        compute_nx(s0["contig_lengths"], 0.90),
        max(s0["contig_lengths"]) if s0["contig_lengths"] else 0,
        int(sum(s0["contig_lengths"]) / max(len(s0["contig_lengths"]), 1)),
    ]
    vals_1   = [
        compute_nx(s1["contig_lengths"], 0.50),
        compute_nx(s1["contig_lengths"], 0.90),
        max(s1["contig_lengths"]) if s1["contig_lengths"] else 0,
        int(sum(s1["contig_lengths"]) / max(len(s1["contig_lengths"]), 1)),
    ]

    x     = np.arange(len(metrics))
    width = 0.35
    b0    = ax.bar(x - width/2, vals_0, width, color=C0,
                   label="No repeats",   edgecolor="white", lw=0.8)
    b1    = ax.bar(x + width/2, vals_1, width, color=C1,
                   label="With repeats", edgecolor="white", lw=0.8)

    for bars in (b0, b1):
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        h + max(vals_0 + vals_1) * 0.01,
                        f"{h:,}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=9)
    ax.set_ylabel("Contig length (bp)", fontsize=9)
    ax.set_title("⑤ N50 / N90 Assembly Quality", fontweight="bold")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, axis="y", ls="--", alpha=0.35)

    # ══════════════════════════════════════════════════════════════════════════
    # [1, 2]  ⑥ Contig-length CDF with N50 / N90 markers
    #
    # The empirical CDF of contig lengths makes N50 and N90 directly readable
    # as horizontal intersections.  A genome that assembles well has a steep
    # CDF (fewer, longer contigs); repeat fragmentation flattens and shifts it.
    # ══════════════════════════════════════════════════════════════════════════
    ax = axes[1, 2]

    x_max_all = 1   # will grow as data is added
    for s, col, lbl in [(s0, C0, "No repeats"), (s1, C1, "With repeats")]:
        cl = sorted(s["contig_lengths"])
        if not cl:
            continue
        cl_arr   = np.array(cl, dtype=float)
        cdf_vals = np.arange(1, len(cl_arr) + 1) / len(cl_arr) * 100

        ax.step(cl_arr, cdf_vals, color=col, lw=2, where="post",
                label=f"Empirical — {lbl}")
        x_max_all = max(x_max_all, cl_arr.max())

        # N50 and N90 markers
        for nx_val, pct, mk, ms in [
            (compute_nx(cl, 0.50), 50, "o", 55),
            (compute_nx(cl, 0.90), 90, "^", 55),
        ]:
            ax.scatter([nx_val], [pct], color=col, s=ms,
                       marker=mk, zorder=5, edgecolors="white", lw=0.7)
            ax.annotate(f"N{'50' if pct==50 else '90'}={nx_val:,}",
                        xy=(nx_val, pct),
                        xytext=(nx_val + x_max_all * 0.04, pct - 6),
                        fontsize=7, color=col,
                        arrowprops=dict(arrowstyle="->", color=col,
                                        lw=0.6, connectionstyle="arc3,rad=0.0"))

    # N50 / N90 reference lines
    for pct, lbl_txt in [(50, "N50 = 50%"), (90, "N90 = 90%")]:
        ax.axhline(pct, color="#CCCCCC", lw=1, ls=":")
        ax.text(x_max_all * 1.01, pct + 1,
                lbl_txt, fontsize=7, color="#AAAAAA", va="bottom")

    ax.set_xlabel("Contig length (bp)", fontsize=9)
    ax.set_ylabel("Cumulative fraction (%)", fontsize=9)
    ax.set_title("⑥ Contig-length CDF", fontweight="bold")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 106)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, ls="--", alpha=0.35)

    # ── Save ───────────────────────────────────────────────────────────────────
    plt.tight_layout()
    plt.savefig(output_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nExtended summary saved to '{output_png}'")


# =============================================================================
# MAIN
# =============================================================================
